# 04 - Baseline B3: árbol y bosque aleatorio

La búsqueda y evaluación de finalistas ya están cerradas. Este cuaderno presenta los CSV versionados y la configuración común seleccionada; no ejecuta Optuna ni reentrena modelos. Los artefactos `.joblib` no están en `results/baseline3/` y deben buscarse en Drive antes de proponer un ajuste.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display
repo = Path.cwd()
if not (repo / 'results' / 'baseline3').exists():
    repo = Path('/content/repo')
RESULTS_DIR = repo / 'results' / 'baseline3'


## Configuración común seleccionada

Bosque aleatorio: `criterion=gini`, `max_depth=12`, `min_samples_leaf=20`, `n_estimators=25`, `max_features=0.5`. Su macro-AUC interno de selección es 0.7901. Ese valor solo eligió la configuración compartida; los resultados de desempeño se mantienen separados por escenario.

In [ ]:
for escenario in ('A', 'B', 'C'):
    ruta = RESULTS_DIR / f'b3_val_{escenario}.csv'
    print(f'Escenario {escenario}')
    if ruta.exists():
        datos = pd.read_csv(ruta)
        display(datos.round(4))
    else:
        print('No está disponible:', ruta)


## Artefactos y límites

Los CSV de validación documentan los reajustes con todo `train`, pero esta carpeta no contiene los modelos serializados. No se repite la búsqueda y no se usa `test`. La selección se basó en valores observados, sin bootstrap emparejado ni afirmación de significancia estadística.

## Recuperar y verificar modelos B3

Tras montar Drive, busca cada artefacto seleccionado, verifica hiperparámetros y lo copia a `export_repo/baseline3`. No ajusta modelos. El campo de escenario y la versión se muestran para revisar su procedencia.

In [ ]:
from pathlib import Path
import sys, shutil, hashlib, joblib
from google.colab import drive
if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
root = Path('/content/drive/MyDrive/Applied-AI-Agent-Project/model_checkpoints')
BUSCAR_MODELOS_B3 = False
if BUSCAR_MODELOS_B3:
    dest = root / 'export_repo' / 'baseline3'; dest.mkdir(parents=True, exist_ok=True)
    esperado = {'criterion': 'gini', 'max_depth': 12, 'min_samples_leaf': 20,
                'n_estimators': 25, 'max_features': 0.5}
    for escenario in ('A', 'B', 'C'):
        nombre = f'b3_{escenario}_B3_bosque_log_loss.joblib'
        encontrados = list((root / 'b3').rglob(nombre))
        if len(encontrados) != 1:
            print(nombre, ': se esperaba 1 archivo; encontrados:', encontrados); continue
        src = encontrados[0]; item = joblib.load(src)
        assert item.get('escenario') == escenario and len(item['features']) == 62
        parametros = item['model'].get_params()
        assert all(parametros[k] == v for k, v in esperado.items()), parametros
        assert item.get('version_sklearn') == '1.7.2'
        target = dest / nombre
        if target.exists():
            print('Ya existe; no se sobrescribe:', target); continue
        shutil.copy2(src, target)
        print('Copiado:', target, '| SHA256:', hashlib.file_digest(target.open('rb'), 'sha256').hexdigest(),
              '| sklearn:', item.get('version_sklearn'))
